# 01 — Data preparation

Create or reuse the project's single stratified 70/30 split, audit class counts, and visualize the preprocessing paths. The split manifests are persistent: rerunning this notebook does not reshuffle an existing split.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import discover_images, prepare_splits
from applied_ai_midterm.transforms import (
    SRGANPairTransform,
    classifier_transform,
    denormalize_classifier,
    denormalize_srgan,
)

config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
raw_dir = PROJECT_ROOT / 'data' / 'raw'
split_dir = PROJECT_ROOT / 'data' / 'splits'

## Discover classes and preserve the split

In [ ]:
all_images = discover_images(raw_dir)
train_manifest, test_manifest = prepare_splits(
    raw_dir,
    split_dir,
    train_ratio=config.train_ratio,
    random_seed=config.random_seed,
)

print('All source images by class:')
display(all_images.groupby(['class_name', 'label']).size().rename('count').to_frame())
print(f'Train: {len(train_manifest):,} | Test: {len(test_manifest):,}')
display(
    train_manifest.groupby('class_name').size().rename('train').to_frame()
    .join(test_manifest.groupby('class_name').size().rename('test'))
)

## Original training samples

In [ ]:
sample_rows = (
    train_manifest.groupby('class_name', group_keys=False)
    .head(2)
    .reset_index(drop=True)
)
sample_images = [
    Image.open(raw_dir / row.filepath).convert('RGB')
    for row in sample_rows.itertuples()
]

fig, axes = plt.subplots(1, len(sample_images), figsize=(12, 3))
for axis, image, row in zip(axes, sample_images, sample_rows.itertuples(), strict=True):
    axis.imshow(image)
    axis.set_title(row.class_name)
    axis.axis('off')
plt.suptitle('Original samples from the training partition')
plt.tight_layout()

## Augmented, ImageNet-normalized classifier inputs

The display helper reverses ImageNet normalization. Augmentation is used only for training; validation/test preprocessing is deterministic.

In [ ]:
classifier_training_transform = classifier_transform(
    config.high_resolution_size, training=True
)
classifier_samples = [classifier_training_transform(image) for image in sample_images]

fig, axes = plt.subplots(1, len(classifier_samples), figsize=(12, 3))
for axis, tensor, row in zip(
    axes, classifier_samples, sample_rows.itertuples(), strict=True
):
    axis.imshow(denormalize_classifier(tensor).permute(1, 2, 0))
    axis.set_title(f'{row.class_name}: 128×128')
    axis.axis('off')
plt.suptitle('Classifier training transformations (display-denormalized)')
plt.tight_layout()

## Paired SRGAN inputs and targets

SRGAN tensors use a separate `[-1, 1]` range. Both rows below are reversed to `[0, 1]` only for display.

In [ ]:
srgan_pair_transform = SRGANPairTransform(
    config.low_resolution_size,
    config.high_resolution_size,
    training=False,
)
srgan_pairs = [srgan_pair_transform(image) for image in sample_images]

fig, axes = plt.subplots(2, len(srgan_pairs), figsize=(12, 6))
for column, ((low_res, high_res), row) in enumerate(
    zip(srgan_pairs, sample_rows.itertuples(), strict=True)
):
    axes[0, column].imshow(denormalize_srgan(low_res).permute(1, 2, 0))
    axes[0, column].set_title(f'{row.class_name}: 32×32 LR')
    axes[1, column].imshow(denormalize_srgan(high_res).permute(1, 2, 0))
    axes[1, column].set_title('128×128 real target')
    axes[0, column].axis('off')
    axes[1, column].axis('off')
plt.suptitle('Aligned SRGAN low/high-resolution pairs')
plt.tight_layout()